# media

> Make the asset files of profile media: copy a file by its content hash, cut a clip or a still with ffmpeg, and find a phrase in a transcript.

A profile media record names a local file as `asset:<sha256>.<ext>`, and the host serves the file from its own origin. This module makes those files. It does not download anything: fossick gets the source files.

`clip` and `still` run ffmpeg. The module imports with no ffmpeg, and these two functions raise `RuntimeError` when ffmpeg is not on the `PATH`.

In [ ]:
#| default_exp media

In [ ]:
#| hide
import json, tempfile
from fastcore.test import *

In [ ]:
#| export
import hashlib, re, shutil, tempfile, unicodedata
from fastcore.utils import *
from fastcore.xtras import run
from rupaka.spec import MEDIA_EXTS

## Assets

`import_asset` copies a file into a store folder. The new name is the SHA-256 of the content and the file type in lower case, so the same content always gets the same name. The file type must be one of `MEDIA_EXTS`.

In [ ]:
#| export
def import_asset(path, store) -> str:
    "Copy the file at `path` into the folder `store`, named by the SHA-256 of its content; give its `asset:<sha256>.<ext>` src."
    p, store = Path(path), Path(store)
    ext, exts = p.suffix[1:].lower(), sorted({x for xs in MEDIA_EXTS.values() for x in xs})
    if ext not in exts: raise ValueError(f'{p.name}: the file type "{ext}" is not one of: {", ".join(exts)}')
    name = f'{hashlib.sha256(p.read_bytes()).hexdigest()}.{ext}'
    store.mkdir(parents=True, exist_ok=True)
    if not (store/name).exists(): shutil.copyfile(p, store/name)
    return f'asset:{name}'

Two files with the same content give one asset:

In [ ]:
d = Path(tempfile.mkdtemp())
(d/'Tha.WAV').write_bytes(b'RIFF tha'); (d/'copy.wav').write_bytes(b'RIFF tha')
a = import_asset(d/'Tha.WAV', d/'store')
test_eq(import_asset(d/'copy.wav', d/'store'), a)
test_eq([p.name for p in (d/'store').iterdir()], [a[6:]])
a

## Clips and stills

`clip` cuts `start` to `end` seconds of a file and stores the result with `import_asset`. Audio becomes AAC in `m4a`, or PCM in `wav` with `ext='wav'`. Video becomes H.264 in `mp4` with AAC sound, and the index goes at the start (`faststart`), so a browser can play it before the whole file loads. The size stays the same, made even for H.264, and never gets larger. `still` stores one frame as a JPEG.

In [ ]:
#| export
def _ffmpeg(*args):
    "Run ffmpeg with `args`; raise `RuntimeError` when ffmpeg is not on the `PATH`."
    if not shutil.which('ffmpeg'): raise RuntimeError('ffmpeg is not on the PATH; install it to cut clips and stills')
    run(['ffmpeg', '-v', 'error', '-nostdin', '-y', *map(str, args)])

def _made(out_dir, ext, *args) -> str:
    "Let ffmpeg write a temporary `ext` file with `args`, then store it in `out_dir` with `import_asset`."
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=out) as d:
        tmp = Path(d)/f'cut.{ext}'
        _ffmpeg(*args, tmp)
        return import_asset(tmp, out)

_ENC = dict(m4a=('-vn', '-c:a', 'aac', '-b:a', '160k'), wav=('-vn', '-c:a', 'pcm_s16le'),
            mp4=('-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-vf', 'scale=trunc(iw/2)*2:trunc(ih/2)*2', '-c:a', 'aac', '-movflags', '+faststart'))

def clip(
    src_file, # an audio or video file that ffmpeg reads
    start:float, # the first second to keep
    end:float, # the second to stop at, more than `start`
    out_dir, # the asset store
    kind:str='audio', # `audio` or `video`
    ext:str=None # for audio: `m4a` (the default) or `wav`
) -> str:
    "Cut `start` to `end` seconds of `src_file` into the asset store `out_dir`; give its `asset:` src."
    if not 0 <= start < end: raise ValueError(f'needs 0 <= start < end, not {start} and {end}')
    if kind not in ('audio', 'video'): raise ValueError(f'kind must be audio or video, not {kind}')
    ext = 'mp4' if kind == 'video' else ext or 'm4a'
    if ext not in ('m4a', 'wav', 'mp4'): raise ValueError(f'an audio clip is m4a or wav, not {ext}')
    return _made(out_dir, ext, '-ss', start, '-i', src_file, '-t', end - start, *_ENC[ext])

def still(src_file, t:float, out_dir) -> str:
    "Store the frame at `t` seconds of the video `src_file` as a JPEG in the asset store `out_dir`; give its `asset:` src."
    return _made(out_dir, 'jpg', '-ss', t, '-i', src_file, '-frames:v', 1, '-q:v', 2)

With ffmpeg, a one-second clip of a generated tone has a duration of one second. This cell needs ffmpeg, so it has the `slow` flag and `nbdev-test` runs it only with `--flags slow`.

In [ ]:
#| slow
_ffmpeg('-f', 'lavfi', '-i', 'sine=frequency=440:duration=3', d/'tone.wav')
c = clip(d/'tone.wav', 1, 2, d/'store')
float(json.loads(run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'json', str(d/'store'/c[6:])]))['format']['duration'])

## Find a phrase in a transcript

`find_span` gives the seconds of a phrase in a transcript, for example to cut a clip of one sentence. A transcript can be WebVTT text, a YouTube `json3` dict, or a Whisper result dict with `segments`, each with optional `words`, or with `words` at the top. A line or a segment with no word times shares its time evenly among its words.

The match ignores case and punctuation. The words of the phrase must come in order. Other words can come between them, at most as many as the phrase has, so a filler such as "um" does not stop a match. The match with the fewest other words wins, and then the first one. `pad` widens the span on each side, and the start is never below 0. With no match, the result is `None`.

In [ ]:
#| export
def _words(s:str) -> list:
    "The words of `s` in lower case, with no punctuation."
    return ''.join(c for c in unicodedata.normalize('NFC', s).lower() if not unicodedata.category(c).startswith('P')).split()

def _spread(text, a, b) -> list:
    "The words of `text` as `(word, start, end)`, each with an even share of `a` to `b` seconds."
    ws, b = _words(text), max(a, b)
    return [(w, a + (b - a) * i / len(ws), a + (b - a) * (i + 1) / len(ws)) for i, w in enumerate(ws)]

_CUE = re.compile(r'(?:(\d+):)?(\d\d):(\d\d)[.,](\d{3})\s*-->\s*(?:(\d+):)?(\d\d):(\d\d)[.,](\d{3})')
def _secs(h, m, s, ms): return int(h or 0) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000

def _vtt(text) -> list:
    "The timed words of WebVTT `text`. Tags such as `<c>` go."
    out = []
    for cue in re.split(r'\n\s*\n', text.replace('\r', '')):
        ls = cue.strip().split('\n')
        i = next((i for i, l in enumerate(ls) if _CUE.search(l)), None)
        if i is None: continue
        g = _CUE.search(ls[i]).groups()
        out += _spread(re.sub(r'<[^>]*>', '', ' '.join(ls[i + 1:])), _secs(*g[:4]), _secs(*g[4:]))
    return out

def _json3(d) -> list:
    "The timed words of a YouTube `json3` dict: each seg lasts until the next seg, and the last one until the end of its event."
    out = []
    for e in d.get('events') or []:
        t0, segs = e.get('tStartMs', 0) / 1000, e.get('segs') or []
        ts = [t0 + s.get('tOffsetMs', 0) / 1000 for s in segs] + [t0 + e.get('dDurationMs', 0) / 1000]
        for s, a, b in zip(segs, ts, ts[1:]): out += _spread(s.get('utf8', ''), a, b)
    return out

def _whisper(d) -> list:
    "The timed words of a Whisper result: the `words` of each segment, else its `text`; or the top-level `words`."
    ws = lambda xs: [x for w in xs if w.get('start') is not None and w.get('end') is not None for x in _spread(w.get('word', ''), w['start'], w['end'])]
    out = [x for s in d.get('segments') or [] for x in (ws(s['words']) if s.get('words') else _spread(s.get('text', ''), s['start'], s['end']))]
    return out or ws(d.get('words') or [])

def find_span(
    transcript, # WebVTT text, a YouTube json3 dict, or a Whisper result dict
    phrase:str, # the words to find
    pad:float=1.0 # seconds to add on each side
) -> tuple|None:
    "The `(start, end)` seconds of the best match of `phrase` in `transcript`, widened by `pad`; None when there is no match."
    ws = _vtt(transcript) if isinstance(transcript, str) else _json3(transcript) if 'events' in transcript else _whisper(transcript)
    p = _words(phrase)
    def end(i):
        j = i
        for w in p[1:]:
            j = next((x for x in range(j + 1, min(len(ws), i + 2 * len(p))) if ws[x][0] == w), None)
            if j is None: return None
        return j
    hits = [(j - i, i, j) for i, (w, *_) in enumerate(ws) if p and w == p[0] and (j := end(i)) is not None]
    if not hits: return None
    _, i, j = min(hits)
    return round(max(0, ws[i][1] - pad), 3), round(ws[j][2] + pad, 3)

In WebVTT, the five words of the first cue share its two seconds. The match ignores the case, the comma and the `<c>` tag:

In [ ]:
vtt = '''WEBVTT

00:00:01.000 --> 00:00:03.000
The next stroke is <c>nam</c>.

00:00:03.000 --> 00:00:05.000
Listen: it rings, and then it fades!'''
test_eq(find_span(vtt, 'Stroke, is nam', pad=0), (1.8, 3.0))
find_span(vtt, 'stroke is nam')

Words can come between the words of the phrase, but the order counts. "it fades" skips "then", and "nam stroke" has no match:

In [ ]:
test_eq(find_span(vtt, 'nam stroke'), None)
find_span(vtt, 'rings it fades', pad=0)

A Whisper result with word times gives the times of the words:

In [ ]:
w = {'segments': [{'start': 1.0, 'end': 3.0, 'text': ' The stroke is nam.', 'words': [{'word': ' The', 'start': 1.0, 'end': 1.3},
       {'word': ' stroke', 'start': 1.8, 'end': 2.2}, {'word': ' is', 'start': 2.3, 'end': 2.4}, {'word': ' nam.', 'start': 2.5, 'end': 2.9}]}]}
find_span(w, 'is nam', pad=0.5)